<a href="https://colab.research.google.com/github/yuviyadav-git/Chunking/blob/main/Semantic_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Statistical Chunking
Breaks the document into sentences, then uses cosine similarity between consecutive sentences to decide whether they belong in the same chunk.

Unlike Consecutive/Cumulative chunking, the similarity threshold is not set manually — it's automatically calculated from the document's own similarity distribution, so it adapts per-document and requires less manual tuning.

Chunks are also capped by a maximum token limit as a fallback: even if similarity stays high, a chunk will be cut once it hits this limit.

In [44]:
!pip install -qU "semantic-router[local]" datasets semantic-chunkers

In [45]:
from datasets import load_dataset

data = load_dataset("jamescalam/ai-arxiv2", split="train")
data


Dataset({
    features: ['id', 'title', 'summary', 'source', 'authors', 'categories', 'comment', 'journal_ref', 'primary_category', 'published', 'updated', 'content', 'references'],
    num_rows: 2673
})

In [46]:
content = data[3]["content"]
print(content[:1000])


# Mamba: Linear-Time Sequence Modeling with Selective State Spaces
# Albert Gu*1 and Tri Dao*2
1Machine Learning Department, Carnegie Mellon University 2Department of Computer Science, Princeton University agu@cs.cmu.edu, tri@tridao.me
# Abstract
Foundation models, now powering most of the exciting applications in deep learning, are almost universally based on the Transformer architecture and its core attention module. Many subquadratic-time architectures such as linear attention, gated convolution and recurrent models, and structured state space models (SSMs) have been developed to address Transformersâ computational ineï¬ciency on long sequences, but they have not performed as well as attention on important modalities such as language. We identify that a key weakness of such models is their inability to perform content-based reasoning, and make several improvements. First, simply letting the SSM parameters be functions of the input addresses their weakness with discrete modalities

In [47]:
content = content[:20_000]

In [48]:
from semantic_router.encoders import HuggingFaceEncoder

encoder = HuggingFaceEncoder(name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [49]:
from semantic_chunkers import StatisticalChunker

chunker = StatisticalChunker(encoder=encoder)

In [50]:
chunks = chunker(docs=[content])

2026-09-24 05:41:24 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 300. Splitting to sentences before semantically merging.


  0%|          | 0/6 [00:00<?, ?it/s]

In [51]:
chunker.print(chunks[0])

Split 1, tokens 300, triggered by: token limit
# Mamba: Linear-Time Sequence Modeling with Selective State Spaces # Albert Gu*1 and Tri Dao*2 1Machine Learning Department, Carnegie Mellon University 2Department of Computer Science, Princeton University agu@cs.cmu.edu, tri@tridao.me # Abstract Foundation models, now powering most of the exciting applications in deep learning, are almost universally based on the Transformer architecture and its core attention module. Many subquadratic-time architectures such as linear attention, gated convolution and recurrent models, and structured state space models (SSMs) have been developed to address Transformersâ computational ineï¬ ciency on long sequences, but they have not performed as well as attention on important modalities such as language. We identify that a key weakness of such models is their inability to perform content-based reasoning, and make several improvements. First, simply letting the SSM parameters be functions of the input addr

# Consecutive Chunking
Works the same way as Statistical Chunking — sentences are compared to their immediate neighbor using cosine similarity — but the similarity threshold is set manually instead of being calculated automatically.

This means the same threshold applies regardless of the document's own similarity patterns — less adaptive than Statistical Chunking, butgives you direct, predictable control once you know what threshold works well for your specific data.

Like Statistical Chunking, chunks are still capped by a maximum token limit as a fallback.

In [52]:
from semantic_chunkers import ConsecutiveChunker
chunker = ConsecutiveChunker(encoder=encoder, score_threshold=0.3)

In [53]:
chunks = chunker(docs=[content])

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/328 [00:00<?, ?it/s]

In [54]:
chunker.print(chunks[0])

Split 1, tokens None, triggered by: 0.06
# Mamba:
----------------------------------------------------------------------------------------


Split 2, tokens None, triggered by: 0.08
Linear-Time Sequence Modeling with Selective State Spaces
----------------------------------------------------------------------------------------


Split 3, tokens None, triggered by: 0.08
# Albert Gu*1 and Tri Dao*2
----------------------------------------------------------------------------------------


Split 4, tokens None, triggered by: 0.05
1Machine Learning Department, Carnegie Mellon University 2Department of Computer Science, Princeton University agu@cs.cmu.edu, tri@tridao.me
----------------------------------------------------------------------------------------


Split 5, tokens None, triggered by: 0.15
# Abstract
----------------------------------------------------------------------------------------


Split 6, tokens None, triggered by: 0.12
Foundation models, now powering most of the exciting

# Cumulative Chunking
Like Consecutive Chunking, the similarity threshold is set manually. The key difference is what gets compared: instead of checking each new sentence only against the one immediately before it, Cumulative Chunking checks each new sentence against the accumulated meaning of the entire chunk built so far.

This makes it better at catching gradual topic drift — a sentence that has slowly moved away from the chunk's overall theme gets flagged even if it's still fairly similar to its immediate neighbor, which Consecutive Chunking can miss. The trade-off is more computation, since it has to keep re-comparing against a growing chunk rather than a single pair of sentences.

Like the other two, chunks are still capped by a maximum token limit as a fallback.

In [55]:
from semantic_chunkers import CumulativeChunker

chunker = CumulativeChunker(encoder=encoder, score_threshold=0.3)

In [56]:
chunks = chunker(docs=[content])

  0%|          | 0/329 [00:00<?, ?it/s]

In [57]:
chunker.print(chunks[0])

Split 1, tokens None, triggered by: 0.06
# Mamba:
----------------------------------------------------------------------------------------


Split 2, tokens None, triggered by: 0.08
Linear-Time Sequence Modeling with Selective State Spaces
----------------------------------------------------------------------------------------


Split 3, tokens None, triggered by: 0.08
# Albert Gu*1 and Tri Dao*2
----------------------------------------------------------------------------------------


Split 4, tokens None, triggered by: 0.05
1Machine Learning Department, Carnegie Mellon University 2Department of Computer Science, Princeton University agu@cs.cmu.edu, tri@tridao.me
----------------------------------------------------------------------------------------


Split 5, tokens None, triggered by: 0.15
# Abstract
----------------------------------------------------------------------------------------


Split 6, tokens None, triggered by: 0.09
Foundation models, now powering most of the exciting